# Bioinspired123D Dataset Generation

Rachel K. Luu, 2026

In [ ]:
from openai import OpenAI
import os

os.environ["OPENAI_API_KEY"] = "YOUR TOKEN HERE"
client = OpenAI()
model = "gpt-4o-mini" 

In [ ]:
from scripts.dataset.pipelines import run_diversification, run_reasoning_generation, LLMConfig
from scripts.dataset.pipelines import generate_blender_general_dataset, GeneralDatasetConfig

cfg = LLMConfig(model="gpt-4o-mini", temperature=0.25, sleep_s=2.0)
cfg_reason = LLMConfig(model="gpt-4o-mini", temperature=0.1, sleep_s=2.0)

# Bioinspired Dataset

Generates approx. 600 (12x5x10) diversified variants and 600 reasoned variants.

In [ ]:
run_diversification(
    client=client,
    csv_path = "../data/raw/biodataset_basecodes.csv",
    output_file="../data/processed/bioinspired_diversified.jsonl",
    n_variants=5, # number of variants per sample
    total_runs=10, # number of times to sample
    cfg=cfg,
    prompt_kind="diverse",
)

run_reasoning_generation(
    client=client,
    input_path="../data/processed/bioinspired_diversified.jsonl",
    output_path="../data/processed/bioinspired_reasoned.jsonl",
    reasoning_example_txt_path="../data/reasoningexample.txt",
    cfg=cfg_reason,
)


# BlendNet Dataset

Generates approx. 1,140 (380x3x1) diversified variants and 1,140 reasoned variants.

In [ ]:
run_diversification(
    client=client,
    csv_path="../data/raw/BlendNet_selected_test.csv",
    output_file="../data/processed/BlendNet_diversified.jsonl",
    n_variants=3, # number of variants per sample
    total_runs=1, # number of times to sample
    cfg=cfg,
    prompt_kind="diverse",
)

run_reasoning_generation(
    client=client,
    input_path="../data/processed/BlendNet_diversified.jsonl",
    output_path="../data/processed/BlendNet_reasoned.jsonl",
    reasoning_example_txt_path="../data/reasoningexample.txt",
    cfg=cfg_reason,
)


# General Blender Dataset

Generates approx. 60 primitive (6x10) + 300 transform (15x20) + 600 advanced (30x20) leveled Blender variants - for a total of approx. 960 variants. 

In [ ]:
cfg = GeneralDatasetConfig(
    output_file="../data/processed/general_dataset.jsonl",
    n_primitive=10,
    n_transform=20,
    n_advanced=20,
    temp=0.25,
    model="gpt-4o-mini",
    write_intermediate_files=False,
    delete_intermediates=True,
)

out_path = generate_blender_general_dataset(client=client, cfg=cfg)
print("Wrote:", out_path)

# Quality Check

This step scans all JSONL datasets in ../data/processed, extracts Blender Python scripts, validates them headlessly in Blender, and automatically removes failed entries based on the validation logs.

In [ ]:
from scripts.pipelines import validate_and_filter_all_processed_jsonl

blender_path = "/mnt/c/Users/Rachel/Downloads/blender-4.2.1-windows-x64/blender-4.2.1-windows-x64/blender.exe"

filtered_files = validate_and_filter_all_processed_jsonl(
    processed_dir="../data/processed",
    blender_path=blender_path,
    out_root="../data/qc_outputs",
    add_ground_plane=False,
    debug_extract=True,
)

print("\nFiltered datasets:")
for f in filtered_files:
    print("  ", f)



After automated validation, we do a final manual sanity pass: visually inspect the rendered outputs, flag any renders that still look incorrect, and remove the corresponding entries from the dataset by ID (and optionally delete their extracted scripts and renders).

# Prompt Generation and Assembly

After generating and QC’ing the dataset locally, run this prompt-generation step to convert the validated JSONL into a final training CSV (prompt, answer). The JSONL files are intentionally not tracked in this repo. Though the final finetuning dataset used for Bioinspired3D is provided in ../data/bioinspired3d_dataset_final.csv

In [ ]:
from scripts.dataset.inputgen import jsonl_to_prompt_csv

# BIOINSPIRED 
n = jsonl_to_prompt_csv(
    jsonl_path="../data/qc_outputs/ ####### ",
    csv_output_path="bioinspired_diverse.csv",
    mode="diverse",
    seed=123,
)

n = jsonl_to_prompt_csv(
    jsonl_path="../data/qc_outputs/ ######## ",,
    csv_output_path="bioinspired_reason.csv",
    mode="reason",
    seed=123,
)

# BLENDNET
n = jsonl_to_prompt_csv(
    jsonl_path="../data/qc_outputs/ ####### ",
    csv_output_path="blendnet_diverse.csv",
    mode="diverse",
    seed=123,
)

n = jsonl_to_prompt_csv(
    jsonl_path="../data/qc_outputs/ ######## ",,
    csv_output_path="blendnet_reason.csv",
    mode="reason",
    seed=123,
)

# GENERAL
n = jsonl_to_prompt_csv(
    jsonl_path="../data/qc_outputs/ ######## ",,
    csv_output_path="general.csv",
    mode="general",
)
